<a href="https://colab.research.google.com/github/cdrowley/notebook-demos/blob/main/fix_geojson_from_string.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Setup (make sure to run this)

In [ ]:
!pip install mapclassify > /dev/null

In [ ]:
import geopandas as gpd
import json
import io
import requests
from jsonschema import validate, ValidationError
from shapely.geometry import Polygon

import warnings
warnings.filterwarnings('ignore')

def is_valid_geojson(s: str) -> bool:
    geojson_schema_url = "https://geojson.org/schema/FeatureCollection.json"
    geojson_schema = requests.get(geojson_schema_url).json()
    try:
        geojson_obj = json.loads(s)
        validate(instance=geojson_obj, schema=geojson_schema)
        return True
    except json.JSONDecodeError as e:
        return False, f"Invalid JSON: {e}"
    except ValidationError as e:
        return False, f"Invalid GeoJSON: {e}"

def input_geojson() -> gpd.GeoDataFrame:
  geojson_str = input('Paste the geojson string: ')
  valid = is_valid_geojson(geojson_str)
  if valid:
      print('The GeoJSON is valid.')
      gdf = gpd.read_file(io.StringIO(geojson_str))
      display(gdf.head())
      return gdf
  else:
      print('Invalid GeoJSON (try it on https://geojson.io): ', valid[1])

### Input Geojson

Run the cell then paste your geojson into the text box that pops up

In [ ]:
gdf = input_geojson()

In [ ]:
# visualise to see any issues
gdf.explore()

### Check Geometry

In [ ]:
# check if the polygon is a valid (aka meets the open geospatial consortium (ogc) standards) geometry, this is not the same as the geojson being valid
assert gdf['geometry'].is_valid.all(), 'Invalid Polygon - Go to (Attempt to) Fix Geometry'
assert gdf['geometry'].is_simple.all(), 'Self-intersecting Polygon  - Go to (Attempt to) Fix Geometry'
assert gdf['geometry'].to_crs(3857).area.gt(0).all(), 'Zero-area Polygon  - Go to (Attempt to) Fix Geometry'

### (Attempt to) Fix Geometry

In [ ]:
try_fix = gdf['geometry'].make_valid().buffer(0)

assert try_fix.is_valid.all() and try_fix.is_simple.all() and try_fix.to_crs(3857).area.gt(0).all()

In [ ]:
try_fix.explore()

In [ ]:
geom_type = try_fix.geom_type.iloc[0]

if (try_fix.is_valid.all() and try_fix.is_simple.all() and try_fix.to_crs(3857).area.gt(0).all()):
  if geom_type == 'Polygon':
    try_fix.to_json()
  elif geom_type == 'MultiPolygon':
      print('The geometry is not a standard Polygon, try the next section (OPTIONAL (Attempt to convert) MultiPolygon to Polygon)')
  else:
    print(f"Error, the {geom_type} isn't a Polygon")

### OPTIONAL (Attempt to convert) MultiPolygon to Polygon

Only use this cell if the message above says so

In [ ]:
# # try very small threshold increases (to get a Polygon) -- however be careful this may change the polygon in potentially unwanted ways

# THRESHOLD = 0.0000000001 # if you go above 0.0001, really consider and check the input vs output polygon

# geom = try_fix.buffer(THRESHOLD).union_all().buffer(THRESHOLD)
# if geom.geom_type == 'Polygon':
#   print('All good, move to next step')
# else:
#   print(f'Try increasing the threshold (remove a zero)')

In [ ]:
# uncomment and get the fixed polygon geojson
# gpd.GeoSeries(geom).set_crs(4326).to_json()